<a href="https://colab.research.google.com/github/Ferdaus71/bangladesh-multi-tool-ai-agent/blob/main/bangladesh_multi_tool_ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

!pip -q install -U \
    langchain \
    langchain-classic \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    langchain-openai \
    tavily-python \
    datasets \
    pandas \
    python-dotenv \
    tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/5

In [2]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

import os
import re
import sqlite3
import json
import shutil
from pathlib import Path

import pandas as pd

from datasets import load_dataset

from IPython.display import display, Markdown

from google.colab import userdata

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ============================================================
# CELL 3: PROJECT CONFIGURATION
# ============================================================

PROJECT_NAME = "bangladesh-multi-tool-ai-agent"

BASE_DIR = Path("/content") / PROJECT_NAME

DATA_DIR = BASE_DIR / "data"

RAW_DATA_DIR = DATA_DIR / "raw"

DATABASE_DIR = DATA_DIR / "db"

SRC_DIR = BASE_DIR / "src"

TEST_DIR = BASE_DIR / "tests"

for directory in [
    BASE_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    DATABASE_DIR,
    SRC_DIR,
    TEST_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Project directory:")
print(BASE_DIR)

print("\nProject structure:")
print("""
bangladesh-multi-tool-ai-agent/
│
├── data/
│   ├── raw/
│   └── db/
│
├── src/
│
├── tests/
│
├── app.py
├── requirements.txt
├── .env.example
└── README.md
""")

Project directory:
/content/bangladesh-multi-tool-ai-agent

Project structure:

bangladesh-multi-tool-ai-agent/
│
├── data/
│   ├── raw/
│   └── db/
│
├── src/
│
├── tests/
│
├── app.py
├── requirements.txt
├── .env.example
└── README.md



In [5]:
# ============================================================
# CELL 4: API CONFIGURATION
# ============================================================

try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = None

try:
    TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")
except Exception:
    TAVILY_API_KEY = None


if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

if TAVILY_API_KEY:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY


print(
    "Google API:",
    "Configured ✓" if GOOGLE_API_KEY else "Missing ✗"
)

print(
    "Tavily API:",
    "Configured ✓" if TAVILY_API_KEY else "Missing ✗"
)

Google API: Configured ✓
Tavily API: Configured ✓


In [6]:
# ============================================================
# CELL 5: DATASET CONFIGURATION
# ============================================================

DATASETS = {

    "institutions": {
        "repo_id":
            "Mahadih534/Institutional-Information-of-Bangladesh",

        "table":
            "institutions",

        "csv":
            RAW_DATA_DIR / "institutions.csv",

        "db":
            DATABASE_DIR / "institutions.db"
    },

    "hospitals": {
        "repo_id":
            "Mahadih534/all-bangladeshi-hospitals",

        "table":
            "hospitals",

        "csv":
            RAW_DATA_DIR / "hospitals.csv",

        "db":
            DATABASE_DIR / "hospitals.db"
    },

    "restaurants": {
        "repo_id":
            "Mahadih534/Bangladeshi-Restaurant-Data",

        "table":
            "restaurants",

        "csv":
            RAW_DATA_DIR / "restaurants.csv",

        "db":
            DATABASE_DIR / "restaurants.db"
    }
}


pd.DataFrame([
    {
        "Dataset": name,
        "HuggingFace Repository": config["repo_id"],
        "SQLite Table": config["table"]
    }
    for name, config in DATASETS.items()
])

,Dataset,HuggingFace Repository,SQLite Table
0,institutions,Mahadih534/Institutional-Information-of-Bangla...,institutions
1,hospitals,Mahadih534/all-bangladeshi-hospitals,hospitals
2,restaurants,Mahadih534/Bangladeshi-Restaurant-Data,restaurants


In [7]:
# ============================================================
# CELL 5: DATASET CONFIGURATION
# ============================================================

DATASETS = {

    "institutions": {
        "repo_id":
            "Mahadih534/Institutional-Information-of-Bangladesh",

        "table":
            "institutions",

        "csv":
            RAW_DATA_DIR / "institutions.csv",

        "db":
            DATABASE_DIR / "institutions.db"
    },

    "hospitals": {
        "repo_id":
            "Mahadih534/all-bangladeshi-hospitals",

        "table":
            "hospitals",

        "csv":
            RAW_DATA_DIR / "hospitals.csv",

        "db":
            DATABASE_DIR / "hospitals.db"
    },

    "restaurants": {
        "repo_id":
            "Mahadih534/Bangladeshi-Restaurant-Data",

        "table":
            "restaurants",

        "csv":
            RAW_DATA_DIR / "restaurants.csv",

        "db":
            DATABASE_DIR / "restaurants.db"
    }
}


pd.DataFrame([
    {
        "Dataset": name,
        "HuggingFace Repository": config["repo_id"],
        "SQLite Table": config["table"]
    }
    for name, config in DATASETS.items()
])

,Dataset,HuggingFace Repository,SQLite Table
0,institutions,Mahadih534/Institutional-Information-of-Bangla...,institutions
1,hospitals,Mahadih534/all-bangladeshi-hospitals,hospitals
2,restaurants,Mahadih534/Bangladeshi-Restaurant-Data,restaurants


In [10]:
import pandas as pd
import sqlite3
from datasets import load_dataset
from IPython.display import display, Markdown

# ============================================================
# CELL 7: DATASET INSPECTION
# ============================================================

# Initialize an empty dictionary to store loaded dataframes
dataframes = {}

for name, config in DATASETS.items():
    # Load the dataset from HuggingFace
    dataset = load_dataset(config["repo_id"], split="train")

    # Convert the dataset to a pandas DataFrame
    df = pd.DataFrame(dataset)

    # Store the dataframe
    dataframes[name] = df

    # Save the raw data to CSV
    df.to_csv(config["csv"], index=False)

    # Connect to the SQLite database
    conn = sqlite3.connect(config["db"])

    # Write the DataFrame to a SQLite table
    df.to_sql(config["table"], conn, if_exists="replace", index=False)

    # Close the connection
    conn.close()


print("Datasets loaded and saved to CSV and SQLite successfully.")

# Now iterate through the loaded dataframes for inspection
for name, dataframe in dataframes.items():

    print("\n")
    print("=" * 80)
    print(f"{name.upper()} DATASET")
    print("=" * 80)

    print(
        "Shape:",
        dataframe.shape
    )

    print("\nColumns:")

    for column in dataframe.columns:

        print(
            f"  • {column}"
        )

    display(
        dataframe.head(5)
    )

README.md:   0%|          | 0.00/850 [00:00<?, ?B/s]

data.csv: reconstructing file:   0%|          |  0.00B / 8.64MB            

data.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/34901 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

...

,INSTITUTE NAME,EIIN,INSTITUTE_TYPE,DIVISION_ID,DIVISION,DISTRICT_ID,DISTRICT,THANA_ID,THANA,UNION_ID,...,AREA_STATUS,GEOGRPYCAL_STATUS,ADDRESS,POST,MANAGEMENT_TYPE,MOBILE,STUDENT_TYPE,EDUCATION_LEVEL,AFFILIATION,MPO_STATUS
0,"ALHAJ MD. SHAMIM AHSAN DAKHIL MADRASAH, MOHISD...",100099,Madrasha,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040913,...,RURAL,COASTAL AREA,MOHISDANGA,CHALAVANGA,NON-GOVERNMENT,1712444536,CO-EDUCATION JOINT,Dakhil,RECOGNIZE,NO
1,AMRAGACHIA SHALEHIA DAKHIL AMDRASAH,100085,Madrasha,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040987,...,RURAL,PLAIN LAND,AMRAGACHIA,HATCHUNAKHALI,NON-GOVERNMENT,1724060685,CO-EDUCATION JOINT,Dakhil,RECOGNIZE,YES
2,GOVT. AMTALI DEGREE COLLEGE,100112,College,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040906,...,UPZILA SADAR MUNICIPALITY,RIVER SIDE/CHAR,"COLLEGE ROAD, AMTALI",AMTALI,GOVERNMENT,1749718415,CO-EDUCATION JOINT,Degree (Pass),RECOGNIZE,NO
3,AMTALI A.K. PILOT HIGH SCHOOL,100011,School,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040947,...,UPZILA SADAR MUNICIPALITY,COASTAL AREA,"437, A K SCHOOL ROAD, AMTALI",AMTALI,NON-GOVERNMENT,1746091667,CO-EDUCATION JOINT,Secondary,RECOGNIZE,YES
4,AMTALI BONDER HOSAINIA FAZIL MADRASHA,100069,Madrasha,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040904,...,OTHER MUNICIPALITY AREA,PLAIN LAND,AMTALI,AMTALI,NON-GOVERNMENT,1714599363,CO-EDUCATION JOINT,Fazil,RECOGNIZE,YES




HOSPITALS DATASET
Shape: (38886, 13)

Columns:
  • Id
  • Name
  • Name (Bangla)
  • Code
  • Agency
  • Type
  • Division
  • District
  • City Corporation
  • Upazila
  • Paurasava
  • Union
  • Private


,Id,Name,Name (Bangla),Code,Agency,Type,Division,District,City Corporation,Upazila,Paurasava,Union,Private
0,1,Dhaka Divisional Health Office,"পরিচালক (স্বাস্থ্য) এর কার্যালয়, ঢাকা বিভাগ, ঢাকা",10000001.0,DGHS,Divisional Level Office,Dhaka,Dhaka,Dhaka South City Corporation,Motijheel,NaN,NaN,0
1,2,Directorate General of Health Services (DGHS),"স্বাস্থ্য অধিদপ্তর, মহাখালী",10000002.0,DGHS,Directorate General or Directorate,Dhaka,Dhaka,Dhaka North City Corperation,Banani,NaN,NaN,0
2,3,"Institute of Epidemiology, Disease Control & R...","রোগতত্ব, রোগ নিয়ন্ত্রন ও গবেষণা ইনষ্টিটিউট ( আ...",10000003.0,DGHS,Public Health Institution,Dhaka,Dhaka,Dhaka North City Corperation,Banani,NaN,NaN,0
3,4,Institute of Public Health (IPH),জনস্বাস্থ্য ইনস্টিটিউট,10000004.0,DGHS,Public Health Institution,Dhaka,Dhaka,Dhaka North City Corperation,Banani,NaN,NaN,0
4,5,Institute Of Public Health Nutrition (IPHN),জনস্বাস্থ্য পুষ্টি প্রতিষ্ঠান,10000005.0,DGHS,Public Health Institution,Dhaka,Dhaka,NaN,NaN,NaN,NaN,0




RESTAURANTS DATASET
Shape: (12703, 8)

Columns:
  • place_id
  • name
  • latitude
  • longitude
  • rating
  • number_of_reviews
  • affluence
  • address


,place_id,name,latitude,longitude,rating,number_of_reviews,affluence,address
0,ChIJx1i4PyCtqjARq5eQI4YeUFE,"Jamal Store, Joykul Bazaar",22.604275,90.094718,0.0,NaN,NaN,"Unnamed Road, Kawkhali, Bangladesh"
1,ChIJjyA9oZytqjAR6apb48G7hSY,Salma Varaitis Store,22.619158,90.105594,5.0,1.0,NaN,"Kawkhali bowlakanda, কাউখালি, Bangladesh"
2,ChIJFYwq-zkLADoRf_tn0mu_rOQ,হাজী বিরিয়ানি হাউজ,22.289046,89.958509,5.0,1.0,NaN,"Charkhali - Mathbaria – Patharghata Rd, Mathba..."
3,ChIJPYyqnw8LADoRycl3-GrLje0,নিউ মুসলিম সুইটস এণ্ড বেকারি,22.288710,89.958482,5.0,4.0,NaN,"সদর রোড, Mathbaria, Bangladesh"
4,ChIJXU_rTB8LADoRYdOJ2LC_Vo4,মেসার্স সততা হোটেল এন্ড রেস্টুরেন্ট,22.286784,89.958116,0.0,NaN,NaN,"7XP5+P69, Mathbaria, Bangladesh"


In [11]:
# ============================================================
# CELL 8: COLUMN NORMALIZATION
# ============================================================

def clean_column_name(
    name: str
):

    name = str(name)

    name = name.strip().lower()

    name = re.sub(
        r"[^a-z0-9]+",
        "_",
        name
    )

    name = name.strip("_")

    if not name:
        name = "column"

    return name


def make_unique_columns(
    columns
):

    used = {}

    result = []

    for column in columns:

        base = clean_column_name(
            column
        )

        count = used.get(
            base,
            0
        )

        used[base] = count + 1

        if count == 0:

            result.append(
                base
            )

        else:

            result.append(
                f"{base}_{count}"
            )

    return result


for name, dataframe in dataframes.items():

    dataframe.columns = (
        make_unique_columns(
            dataframe.columns
        )
    )


for name, dataframe in dataframes.items():

    print(
        f"\n{name}:"
    )

    print(
        list(dataframe.columns)
    )


institutions:
['institute_name', 'eiin', 'institute_type', 'division_id', 'division', 'district_id', 'district', 'thana_id', 'thana', 'union_id', 'union_name', 'mauza_id', 'mauza_name', 'area_status', 'geogrpycal_status', 'address', 'post', 'management_type', 'mobile', 'student_type', 'education_level', 'affiliation', 'mpo_status']

hospitals:
['id', 'name', 'name_bangla', 'code', 'agency', 'type', 'division', 'district', 'city_corporation', 'upazila', 'paurasava', 'union', 'private']

restaurants:
['place_id', 'name', 'latitude', 'longitude', 'rating', 'number_of_reviews', 'affluence', 'address']


In [12]:
# ============================================================
# CELL 9: SQLITE TYPE INFERENCE
# ============================================================

def infer_sqlite_type(
    series
):

    if pd.api.types.is_integer_dtype(series):

        return "INTEGER"

    if pd.api.types.is_float_dtype(series):

        return "REAL"

    if pd.api.types.is_bool_dtype(series):

        return "INTEGER"

    return "TEXT"

In [13]:
# ============================================================
# CELL 10: SQLITE DATABASE BUILDER
# ============================================================

def dataframe_to_sqlite(
    dataframe,
    database_path,
    table_name
):

    dataframe = dataframe.copy()

    # Convert NaN → None
    dataframe = dataframe.where(
        pd.notnull(dataframe),
        None
    )

    database_path = Path(
        database_path
    )

    if database_path.exists():

        database_path.unlink()

    connection = sqlite3.connect(
        database_path
    )

    cursor = connection.cursor()

    # --------------------------------------------------------
    # CREATE TABLE
    # --------------------------------------------------------

    column_definitions = []

    for column in dataframe.columns:

        sqlite_type = infer_sqlite_type(
            dataframe[column]
        )

        column_definitions.append(
            f'"{column}" {sqlite_type}'
        )

    create_query = f"""
    CREATE TABLE "{table_name}"
    (
        {", ".join(column_definitions)}
    )
    """

    cursor.execute(
        create_query
    )

    # --------------------------------------------------------
    # INSERT DATA
    # --------------------------------------------------------

    columns = ", ".join(
        f'"{column}"'
        for column in dataframe.columns
    )

    placeholders = ", ".join(
        ["?"] * len(dataframe.columns)
    )

    insert_query = f"""
    INSERT INTO "{table_name}"
    ({columns})
    VALUES ({placeholders})
    """

    rows = [
        tuple(row)
        for row in dataframe.itertuples(
            index=False,
            name=None
        )
    ]

    cursor.executemany(
        insert_query,
        rows
    )

    connection.commit()

    connection.close()

    print(
        f"✓ {table_name}.db created"
    )

    print(
        f"  Rows: {len(dataframe):,}"
    )

    print(
        f"  Columns: {len(dataframe.columns)}"
    )

In [14]:
# ============================================================
# CELL 11: BUILD SQLITE DATABASES
# ============================================================

for name, config in DATASETS.items():

    dataframe_to_sqlite(

        dataframe=dataframes[name],

        database_path=config["db"],

        table_name=config["table"]
    )


print("\n")
print("=" * 70)
print("ALL SQLITE DATABASES CREATED SUCCESSFULLY")
print("=" * 70)

✓ institutions.db created
  Rows: 34,901
  Columns: 23
✓ hospitals.db created
  Rows: 38,886
  Columns: 13
✓ restaurants.db created
  Rows: 12,703
  Columns: 8


ALL SQLITE DATABASES CREATED SUCCESSFULLY


In [15]:
# ============================================================
# CELL 12: DATABASE VERIFICATION
# ============================================================

def inspect_database(
    database_path,
    table_name
):

    connection = sqlite3.connect(
        database_path
    )

    cursor = connection.cursor()

    # Row count
    cursor.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    )

    count = cursor.fetchone()[0]

    # Schema
    cursor.execute(
        f'PRAGMA table_info("{table_name}")'
    )

    schema = cursor.fetchall()

    connection.close()

    return count, schema


for name, config in DATASETS.items():

    count, schema = inspect_database(
        config["db"],
        config["table"]
    )

    print("\n" + "=" * 70)

    print(
        name.upper()
    )

    print(
        "Rows:",
        f"{count:,}"
    )

    print(
        "\nSchema:"
    )

    for column in schema:

        print(
            f"  {column[1]} → {column[2]}"
        )


INSTITUTIONS
Rows: 34,901

Schema:
  institute_name → TEXT
  eiin → INTEGER
  institute_type → TEXT
  division_id → INTEGER
  division → TEXT
  district_id → INTEGER
  district → TEXT
  thana_id → INTEGER
  thana → TEXT
  union_id → INTEGER
  union_name → TEXT
  mauza_id → INTEGER
  mauza_name → TEXT
  area_status → TEXT
  geogrpycal_status → TEXT
  address → TEXT
  post → TEXT
  management_type → TEXT
  mobile → TEXT
  student_type → TEXT
  education_level → TEXT
  affiliation → TEXT
  mpo_status → TEXT

HOSPITALS
Rows: 38,886

Schema:
  id → INTEGER
  name → TEXT
  name_bangla → TEXT
  code → REAL
  agency → TEXT
  type → TEXT
  division → TEXT
  district → TEXT
  city_corporation → TEXT
  upazila → TEXT
  paurasava → TEXT
  union → TEXT
  private → INTEGER

RESTAURANTS
Rows: 12,703

Schema:
  place_id → TEXT
  name → TEXT
  latitude → REAL
  longitude → REAL
  rating → REAL
  number_of_reviews → REAL
  affluence → REAL
  address → TEXT


In [16]:
# ============================================================
# CELL 13: SAFE DATABASE QUERY ENGINE
# ============================================================

FORBIDDEN_SQL = [

    "insert ",
    "update ",
    "delete ",
    "drop ",
    "alter ",
    "create ",
    "replace ",
    "attach ",
    "detach ",
    "vacuum ",
    "pragma "
]


def validate_sql(
    sql
):

    sql = sql.strip()

    if not sql:

        return False

    lowered = sql.lower()

    # Only SELECT / WITH
    if not lowered.startswith(
        ("select ", "with ")
    ):

        return False

    # Block destructive operations
    for keyword in FORBIDDEN_SQL:

        if keyword in lowered:

            return False

    return True


def execute_readonly_sql(
    database_path,
    sql,
    max_rows=20
):

    if not validate_sql(sql):

        return (
            "Rejected: only read-only "
            "SELECT/WITH queries are allowed."
        )

    database_path = Path(
        database_path
    )

    if not database_path.exists():

        return (
            "Database does not exist."
        )

    # Automatically limit list results
    if " limit " not in sql.lower():

        sql = (
            sql.rstrip(";")
            + f" LIMIT {max_rows}"
        )

    try:

        connection = sqlite3.connect(
            f"file:{database_path}?mode=ro",
            uri=True
        )

        connection.row_factory = (
            sqlite3.Row
        )

        cursor = connection.cursor()

        cursor.execute(sql)

        rows = cursor.fetchall()

        connection.close()

        if not rows:

            return "No matching records found."

        headers = list(
            rows[0].keys()
        )

        output = []

        output.append(
            " | ".join(headers)
        )

        output.append(
            " | ".join(
                ["---"] * len(headers)
            )
        )

        for row in rows[:max_rows]:

            output.append(
                " | ".join(
                    str(
                        row[column]
                    )
                    if row[column] is not None
                    else ""
                    for column in headers
                )
            )

        return "\n".join(
            output
        )

    except Exception as error:

        return (
            f"SQL Error: {error}"
        )

In [17]:
# ============================================================
# CELL 14: LANGCHAIN DATABASE TOOLS
# ============================================================

from langchain_core.tools import tool


def get_database_schema(
    database_path,
    table_name
):

    connection = sqlite3.connect(
        database_path
    )

    cursor = connection.cursor()

    cursor.execute(
        f'PRAGMA table_info("{table_name}")'
    )

    rows = cursor.fetchall()

    connection.close()

    schema = []

    for row in rows:

        schema.append(
            f"{row[1]} ({row[2]})"
        )

    return ", ".join(schema)


def create_database_tool(
    name,
    database_path,
    table_name,
    description
):

    schema = get_database_schema(
        database_path,
        table_name
    )

    @tool(name)
    def database_tool(
        sql: str
    ) -> str:
        """
        Execute a read-only SQL query
        against the selected Bangladesh dataset.
        """

        return execute_readonly_sql(
            database_path,
            sql
        )

    database_tool.description = f"""
{description}

Table:
{table_name}

Schema:
{schema}

Rules:
- Use SELECT/WITH only.
- Never use INSERT, UPDATE, DELETE, DROP or ALTER.
- Use exact schema column names.
- Use COUNT() for count questions.
- Use LIMIT 10 for normal list queries.
"""

    return database_tool

In [18]:
# ============================================================
# CELL 15: CREATE DATABASE TOOLS
# ============================================================

institutions_tool = create_database_tool(

    name="InstitutionsDBTool",

    database_path=
        DATASETS["institutions"]["db"],

    table_name=
        "institutions",

    description="""
Use this tool for institutional information
in Bangladesh.

Examples:
- universities
- colleges
- schools
- institutions
- government institutions
- EIIN
- institution counts
- district
- division
- education level
- institution type
"""
)


hospitals_tool = create_database_tool(

    name="HospitalsDBTool",

    database_path=
        DATASETS["hospitals"]["db"],

    table_name=
        "hospitals",

    description="""
Use this tool for hospital and health-facility
information in Bangladesh.

Examples:
- hospitals
- health institutions
- health facilities
- hospital counts
- hospital type
- agency
- district
- division
- public/private status

Do not invent bed capacity, doctor count,
ICU capacity or other fields unless those
fields exist in the database schema.
"""
)


restaurants_tool = create_database_tool(

    name="RestaurantsDBTool",

    database_path=
        DATASETS["restaurants"]["db"],

    table_name=
        "restaurants",

    description="""
Use this tool for Bangladesh restaurant data.

Examples:
- restaurant names
- restaurant addresses
- restaurant locations
- ratings
- number of reviews
- coordinates
- restaurants by city/district
"""
)


database_tools = [

    institutions_tool,

    hospitals_tool,

    restaurants_tool

]


print(
    "✓ InstitutionsDBTool"
)

print(
    "✓ HospitalsDBTool"
)

print(
    "✓ RestaurantsDBTool"
)

✓ InstitutionsDBTool
✓ HospitalsDBTool
✓ RestaurantsDBTool


In [19]:
# ============================================================
# CELL 16: DATABASE TOOL TESTING
# ============================================================

print(
    "Institutions Tool Test:\n"
)

print(
    institutions_tool.invoke(
        {
            "sql":
                "SELECT * FROM institutions LIMIT 3"
        }
    )
)


print(
    "\n\nHospitals Tool Test:\n"
)

print(
    hospitals_tool.invoke(
        {
            "sql":
                "SELECT * FROM hospitals LIMIT 3"
        }
    )
)


print(
    "\n\nRestaurants Tool Test:\n"
)

print(
    restaurants_tool.invoke(
        {
            "sql":
                "SELECT * FROM restaurants LIMIT 3"
        }
    )
)

Institutions Tool Test:

institute_name | eiin | institute_type | division_id | division | district_id | district | thana_id | thana | union_id | union_name | mauza_id | mauza_name | area_status | geogrpycal_status | address | post | management_type | mobile | student_type | education_level | affiliation | mpo_status
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
ALHAJ MD. SHAMIM AHSAN DAKHIL MADRASAH, MOHISDANGA | 100099 | Madrasha | 10 | BARISAL | 1004 | BARGUNA | 100409 | AMTALI | 10040913 | AMTALI | 10040913712 | NACHNA PARA |  RURAL | COASTAL AREA | MOHISDANGA | CHALAVANGA | NON-GOVERNMENT | 1712444536 | CO-EDUCATION JOINT | Dakhil | RECOGNIZE | NO
AMRAGACHIA SHALEHIA DAKHIL AMDRASAH | 100085 | Madrasha | 10 | BARISAL | 1004 | BARGUNA | 100409 | AMTALI | 10040987 | KUKUA | 10040987307 | DAKSHIN CHUNAKHALI |  RURAL | PLAIN LAND | AMRAGACHIA | HATCHUNAKHALI | NON-GOVERNMENT | 1724060685 | CO-EDU

In [20]:
# ============================================================
# CELL 17: SQL SECURITY TEST
# ============================================================

dangerous_queries = [

    "DELETE FROM institutions",

    "DROP TABLE institutions",

    "UPDATE institutions SET name='test'",

    "INSERT INTO institutions VALUES ('test')",

    "ALTER TABLE institutions ADD COLUMN test TEXT"

]


for query in dangerous_queries:

    result = execute_readonly_sql(

        DATASETS["institutions"]["db"],

        query
    )

    print(
        f"\nQuery: {query}"
    )

    print(
        f"Result: {result}"
    )


Query: DELETE FROM institutions
Result: Rejected: only read-only SELECT/WITH queries are allowed.

Query: DROP TABLE institutions
Result: Rejected: only read-only SELECT/WITH queries are allowed.

Query: UPDATE institutions SET name='test'
Result: Rejected: only read-only SELECT/WITH queries are allowed.

Query: INSERT INTO institutions VALUES ('test')
Result: Rejected: only read-only SELECT/WITH queries are allowed.

Query: ALTER TABLE institutions ADD COLUMN test TEXT
Result: Rejected: only read-only SELECT/WITH queries are allowed.


In [21]:
# ============================================================
# CELL 18: WEB SEARCH TOOL
# ============================================================

from langchain_community.tools.tavily_search import (
    TavilySearchResults
)


if not TAVILY_API_KEY:

    raise ValueError(
        "TAVILY_API_KEY is missing. "
        "Add it to Colab Secrets."
    )


tavily_search = TavilySearchResults(
    max_results=5
)


@tool("WebSearchTool")
def web_search(
    query: str
) -> str:
    """
    Search the web for general, current,
    policy-related or external information
    that is not available in the local databases.
    """

    try:

        results = tavily_search.invoke(
            {
                "query": query
            }
        )

        if not results:

            return (
                "No web results found."
            )

        output = []

        for result in results:

            if isinstance(
                result,
                dict
            ):

                output.append(

                    f"Title: "
                    f"{result.get('title', '')}\n"

                    f"URL: "
                    f"{result.get('url', '')}\n"

                    f"Content: "
                    f"{result.get('content', '')}"
                )

            else:

                output.append(
                    str(result)
                )

        return "\n\n".join(
            output
        )

    except Exception as error:

        return (
            f"Web Search Error: {error}"
        )


print(
    "✓ WebSearchTool created"
)

✓ WebSearchTool created


/tmp/ipykernel_1788/751034254.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import (
/tmp/ipykernel_1788/751034254.py:18: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(


In [22]:
# ============================================================
# CELL 19: TEST WEB SEARCH
# ============================================================

result = web_search.invoke(
    {
        "query":
            "What is the role of DGHS in Bangladesh?"
    }
)

print(result)

Title: Guideline
URL: https://old.dghs.gov.bd/index.php/en/publications/guideline
Content: The DGHS is one of the agencies of the Ministry of Health & Family Welfare of Bangladesh. DGHS stands for Directorate General of Health Services.

Title: Directorate General of Health Services (Bangladesh)
URL: https://en.wikipedia.org/wiki/Directorate_General_of_Health_Services_(Bangladesh)
Content: The Directorate General of Health Services (DGHS) (Bengali: স্বাস্থ্য অধিদপ্তর) is a Bangladeshi government directorate under the Ministry of Health and Family Welfare "Ministry of Health and Family Welfare (Bangladesh)") responsible for health services in Bangladesh.

## History

The DGHS was established as a directorate in 1978. It was upgraded to a directorate general in 1980.

On 23 January 2019, the Anti-Corruption Commission began investigating 23 officers at the DGHS for corruption and recommended their transfers. [...] Wikipedia
The Free Encyclopedia

## Contents

# Directorate General of Hea

In [23]:
# ============================================================
# CELL 20: AGENT SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """

You are the Bangladesh Multi-Tool AI Agent.

Your job is to answer user questions accurately
using the appropriate tool.

====================================================
TOOL ROUTING
====================================================

INSTITUTIONSDBTOOL
------------------

Use for:

- universities
- colleges
- schools
- educational institutions
- government institutions
- EIIN
- institution counts
- institution types
- district
- division
- education level


HOSPITALSDBTOOL
---------------

Use for:

- hospitals
- health institutions
- health facilities
- hospital counts
- hospital type
- agency
- district
- division
- public/private status


RESTAURANTSDBTOOL
-----------------

Use for:

- restaurants
- restaurant names
- restaurant addresses
- ratings
- reviews
- location
- latitude
- longitude
- restaurant count


WEBSEARCHTOOL
-------------

Use for:

- general knowledge
- definitions
- government policies
- laws
- current information
- healthcare policy
- cultural context
- external information
- information unavailable in local databases


====================================================
IMPORTANT RULES
====================================================

1. Never fabricate database information.

2. Always inspect and follow the database schema
   supplied by the selected tool.

3. Database SQL must only use SELECT or WITH.

4. Never generate INSERT, UPDATE, DELETE,
   DROP or ALTER queries.

5. For list questions use LIMIT 10 unless
   the user requests another number.

6. For count questions use COUNT().

7. If a requested field does not exist in the
   database, explicitly say that the supplied
   dataset does not contain that field.

8. Do not invent hospital bed capacity,
   doctor counts, ICU beds or facilities.

9. Prefer local databases for questions that
   are directly answerable from them.

10. Use WebSearchTool for general/current
    knowledge.

11. Answer naturally.

12. Answer in the same language as the user.

"""

In [24]:
# ============================================================
# CELL 21: INITIALIZE GEMINI
# ============================================================

from langchain_google_genai import (
    ChatGoogleGenerativeAI
)


if not GOOGLE_API_KEY:

    raise ValueError(
        "GOOGLE_API_KEY is missing. "
        "Add it to Colab Secrets."
    )


llm = ChatGoogleGenerativeAI(

    model="gemini-2.5-flash",

    temperature=0,

    google_api_key=GOOGLE_API_KEY
)


print(
    "✓ Gemini initialized successfully"
)

✓ Gemini initialized successfully


In [25]:
# ============================================================
# CELL 22: CREATE LANGCHAIN AGENT
# ============================================================

from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent
)


all_tools = [

    institutions_tool,

    hospitals_tool,

    restaurants_tool,

    web_search

]


prompt = ChatPromptTemplate.from_messages([

    (
        "system",
        SYSTEM_PROMPT
    ),

    (
        "human",
        "{input}"
    ),

    (
        "placeholder",
        "{agent_scratchpad}"
    )

])


agent = create_tool_calling_agent(

    llm=llm,

    tools=all_tools,

    prompt=prompt
)


agent_executor = AgentExecutor(

    agent=agent,

    tools=all_tools,

    verbose=True,

    max_iterations=8,

    handle_parsing_errors=True
)


print(
    "✓ Main Agent created successfully"
)

print(
    "✓ AgentExecutor ready"
)

✓ Main Agent created successfully
✓ AgentExecutor ready


In [26]:
# ============================================================
# CELL 23: PROFESSIONAL QUERY FUNCTION
# ============================================================

def ask_agent(
    question: str
):

    if not question.strip():

        return (
            "Please enter a valid question."
        )

    try:

        response = agent_executor.invoke(

            {
                "input": question
            }

        )

        return response["output"]

    except Exception as error:

        return (
            f"Agent Error: {error}"
        )

In [27]:
# ============================================================
# CELL 24: INSTITUTION QUERY
# ============================================================

question = (
    "How many institutions are in Rajshahi?"
)

answer = ask_agent(
    question
)

print(
    "Question:",
    question
)

print(
    "\nAnswer:"
)

print(
    answer
)



> Entering new AgentExecutor chain...

Invoking: `InstitutionsDBTool` with `{'sql': "SELECT COUNT(*) FROM institutions WHERE division = 'Rajshahi'"}`


COUNT(*)
---
0There are no institutions in Rajshahi in the database.

> Finished chain.
Question: How many institutions are in Rajshahi?

Answer:
There are no institutions in Rajshahi in the database.


In [28]:
# ============================================================
# CELL 25: HOSPITAL QUERY
# ============================================================

question = (
    "How many hospitals are in Dhaka?"
)

answer = ask_agent(
    question
)

print(
    "Question:",
    question
)

print(
    "\nAnswer:"
)

print(
    answer
)



> Entering new AgentExecutor chain...

Invoking: `HospitalsDBTool` with `{'sql': "SELECT COUNT(*) FROM hospitals WHERE district = 'Dhaka'"}`


COUNT(*)
---
2919There are 2919 hospitals in Dhaka.

> Finished chain.
Question: How many hospitals are in Dhaka?

Answer:
There are 2919 hospitals in Dhaka.


In [29]:
# ============================================================
# CELL 26: RESTAURANT QUERY
# ============================================================

question = (
    "Find 10 restaurants in Chattogram."
)

answer = ask_agent(
    question
)

print(
    "Question:",
    question
)

print(
    "\nAnswer:"
)

print(
    answer
)



> Entering new AgentExecutor chain...

Invoking: `RestaurantsDBTool` with `{'sql': "SELECT name, address FROM restaurants WHERE address LIKE '%Chattogram%' LIMIT 10"}`


name | address
--- | ---
Maa Studio B | Chattogram 4640, Bangladesh
BITE 365 | Press Club Road, Beside Chandpur Press Club (Opposite of Chandpur Library), Chandpur Sadar, Chattogram, Bangladesh
Reyad Restaurent | Chattogram 4376, Bangladesh
Halda resturent and party Center | fatikchari, Chattogram 4355, Bangladesh
ইকবালের টং | 8QVP+JXW, Chattogram, Bangladesh
Cafe City Extra | 9QCG+WGH, Chattogram, Bangladesh
মুহাঃ সিয়াম | 9Q5F+Q7R, Chattogram 4217, Bangladesh
Shapla Restaurant | 8QWJ+7MP, Chattogram, Bangladesh
সাধের কাচ্চি | Chattogram 4216, Bangladesh
Chef's Kitchen | 8QQV+R2P, Chattogram, BangladeshHere are 10 restaurants in Chattogram:

* Maa Studio B
* BITE 365
* Reyad Restaurent
* Halda resturent and party Center
* ইকবালের টং
* Cafe City Extra
* মুহাঃ সিয়াম
* Shapla Restaurant
* সাধের কাচ্চি
* Chef's Kitchen



In [30]:
# ============================================================
# CELL 27: WEB SEARCH QUERY
# ============================================================

question = (
    "What is the role of DGHS in Bangladesh?"
)

answer = ask_agent(
    question
)

print(
    "Question:",
    question
)

print(
    "\nAnswer:"
)

print(
    answer
)



> Entering new AgentExecutor chain...

Invoking: `WebSearchTool` with `{'query': 'role of DGHS in Bangladesh'}`


Title: Directorate General of Health Services (Bangladesh)
URL: https://en.wikipedia.org/wiki/Directorate_General_of_Health_Services_(Bangladesh)
Content: The Directorate General of Health Services (DGHS) (Bengali: স্বাস্থ্য অধিদপ্তর) is a Bangladeshi government directorate under the Ministry of Health and Family Welfare "Ministry of Health and Family Welfare (Bangladesh)") responsible for health services in Bangladesh.

## History

The DGHS was established as a directorate in 1978. It was upgraded to a directorate general in 1980.

On 23 January 2019, the Anti-Corruption Commission began investigating 23 officers at the DGHS for corruption and recommended their transfers. [...] Abul Bashar Mohammed Khurshid Alam is appointed as director general on 23 July 2020. Alam criticised media coverage on the healthcare system and the Ministry of Health and Family Welfare. The Anti

In [32]:
# ============================================================
# CELL 28: DATASET LIMITATION TEST
# ============================================================

question = (
    "List 10 hospitals in Dhaka with bed capacity."
)

answer = ask_agent(
    question
)

print(
    "Question:",
    question
)

print(
    "\nAgent Response:"
)

print(
    answer
)



> Entering new AgentExecutor chain...
[{'type': 'text', 'text': "I'm sorry, but I cannot provide information on hospital bed capacity", 'extras': {'signature': 'CiIBEU0yD3ACC4gO/GS9g8umOvrkt7kYJAtMCrgXlgOmM/tNCmcBEU0yD2ZDE1YwIkujOnHcnwRWILONMyf49Q7Qghacng1v653rplHOC24iMMxbkQruQ8LZmlOuLWL4U4RAZ2PUdDfNbfbqA4ILZCsFzsoPQe7/h9A95PRRqNSwzrXWLOi16xqO/8+5CuMBARFNMg9MeU15gTXEpSp9BUFEUYmRjUqHtqk5CQ6X4WdJ4ce4QSzY2r3xqRBtoexozwFeiQ4eWOh3H4lZY4ilvqBzDpuzuSUJxUBkJpFBbHAergoYBFw5H23fU9yZAjZmo1yo8G9Mc3BIWW/dsqEkHxZSVHJSoiuV5rupKkMpideEhQQvmr+R90W2ttDLgrSXGk3Z+jWIlvLV/0sAp+zPj0DNCWZzehc9wpjy+AoTcXyimU8CxzcrVXZVTk0GTdiX160W9rZ90Moes8usptqhWa8s99hJddM4vzQ+CGo6NSpW4S8K0QEBEU0yD1yvzmO6kQjAsc0DVAa/v4PKtZhUNgmSCtycDMLuTrrXT01mzk9uG397jYWbulcQOs59MaaY3BxnRAyDGLrnVaCfMPRfM3Vqbn92nEqsde9O8gC8ycjLMD/+yD6dcEDuT+zY7XhEovZscE0UuBxImUDpVMzH/dLvdV4KNr2cF0XmvPnxwg5WUOVNKCsOrFaFUWHGwlte/wJlDTOptOzY9TZdDK9SrWO5OYg1oanTfVIYMUNY+uZIAV8bTug7+LBhxO414SoPXLCT2yHNIQrFAQERTTIPovD4aLLcU7dcC0xk2rwgZ6mNDm9mARY4OEfLxenzJ2q7lxQ/h

In [33]:
# ============================================================
# CELL 29: COMPLETE DEMO
# ============================================================

demo_questions = [

    "How many institutions are in Rajshahi?",

    "List 10 institutions in Dhaka.",

    "How many hospitals are in Dhaka?",

    "List 10 health facilities in Chattogram.",

    "Find 10 restaurants in Chattogram.",

    "Which restaurants have ratings above 4.5?",

    "What is the role of DGHS in Bangladesh?",

    "What is the healthcare policy in Bangladesh?"

]


for index, question in enumerate(
    demo_questions,
    start=1
):

    print(
        "\n" + "=" * 80
    )

    print(
        f"QUERY {index}"
    )

    print(
        "=" * 80
    )

    print(
        "User:",
        question
    )

    print(
        "\nAgent:"
    )

    print(
        ask_agent(question)
    )


QUERY 1
User: How many institutions are in Rajshahi?

Agent:


> Entering new AgentExecutor chain...

Invoking: `InstitutionsDBTool` with `{'sql': "SELECT COUNT(*) FROM institutions WHERE division = 'Rajshahi'"}`


COUNT(*)
---
0There are no institutions listed for Rajshahi in the database.

> Finished chain.
There are no institutions listed for Rajshahi in the database.

QUERY 2
User: List 10 institutions in Dhaka.

Agent:


> Entering new AgentExecutor chain...

Invoking: `InstitutionsDBTool` with `{'sql': "SELECT institute_name, address FROM institutions WHERE district = 'Dhaka' LIMIT 10"}`


No matching records found.
Invoking: `InstitutionsDBTool` with `{'sql': "SELECT institute_name, address FROM institutions WHERE division = 'Dhaka' LIMIT 10"}`


No matching records found.
Invoking: `InstitutionsDBTool` with `{'sql': "SELECT institute_name, address FROM institutions WHERE district LIKE '%Dhaka%' LIMIT 10"}`


institute_name | address
--- | ---
ADABOR IDEAL SCHOOL | ADABOR BAZAR

In [34]:
# ============================================================
# CELL 30: INTERACTIVE CHAT
# ============================================================

print("=" * 70)

print(
    "🇧🇩 BANGLADESH MULTI-TOOL AI AGENT"
)

print("=" * 70)

print(
    "Type 'exit' to stop."
)


while True:

    question = input(
        "\nYou: "
    ).strip()


    if question.lower() in {
        "exit",
        "quit"
    }:

        print(
            "\nGoodbye! 👋"
        )

        break


    if not question:

        continue


    print(
        "\n🤖 Agent:"
    )

    print(
        ask_agent(question)
    )

🇧🇩 BANGLADESH MULTI-TOOL AI AGENT
Type 'exit' to stop.

You: Find 10 restaurants in Chattogram

🤖 Agent:


> Entering new AgentExecutor chain...
Agent Error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 35.4529729s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'gen

In [35]:
# ============================================================
# CELL 31: CREATE GITHUB PROJECT STRUCTURE
# ============================================================

requirements_content = """
langchain>=1.0.0
langchain-classic>=1.0.0
langchain-core>=1.0.0
langchain-community>=0.4.0
langchain-google-genai>=4.0.0
langchain-openai>=1.0.0
tavily-python>=0.7.0
datasets>=3.0.0
pandas>=2.2.0
python-dotenv>=1.0.1
pytest>=8.0.0
""".strip()


env_content = """
GOOGLE_API_KEY=your_google_api_key
TAVILY_API_KEY=your_tavily_api_key
GEMINI_MODEL=gemini-2.5-flash
""".strip()


gitignore_content = """
.venv/
__pycache__/
*.pyc
.env
.ipynb_checkpoints/

data/raw/*.csv
data/db/*.db
""".strip()


(BASE_DIR / "requirements.txt").write_text(
    requirements_content
)

(BASE_DIR / ".env.example").write_text(
    env_content
)

(BASE_DIR / ".gitignore").write_text(
    gitignore_content
)

print(
    "✓ GitHub project configuration created"
)

✓ GitHub project configuration created
